In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

In [3]:
DATA_PATH = "../data/interim/restaurant_clean.csv"

df = pd.read_csv(DATA_PATH)

df["DEMAND_DATE"] = pd.to_datetime(df["DEMAND_DATE"])
df = df.sort_values("DEMAND_DATE").reset_index(drop=True)

df.head()

,DEMAND_DATE,MONDAY,TUESDAY,WEDNESDAY,THURSDAY,FRIDAY,SATURDAY,SUNDAY,MONTH_JAN,MONTH_FEB,...,AIR_TEMPERATURE_NO_DAYS_ABOVE_7D_MEAN,WIND_NO_DAYS_BELOW_7D_MEAN,CLOUD_COVER_NO_DAYS_BELOW_7D_MEAN,PRECIPITATION_NO_DAYS_BELOW_7D_MEAN,SUNSHINE_NO_DAYS_BELOW_7D_MEAN,AIR_TEMPERATURE_NO_DAYS_BELOW_7D_MEAN,ISHOLIDAY,WEEKEND,DAY_OF_WEEK,MONTH
0,2013-10-04,0,0,0,0,1,0,0,0,0,...,3,4,4,0,4,4,0,0,Friday,10
1,2013-10-05,0,0,0,0,0,1,0,0,0,...,3,4,3,6,3,4,0,1,Saturday,10
2,2013-10-06,0,0,0,0,0,0,1,0,0,...,3,4,3,6,3,4,0,1,Sunday,10
3,2013-10-07,1,0,0,0,0,0,0,0,0,...,4,5,3,6,4,3,0,0,Monday,10
4,2013-10-08,0,1,0,0,0,0,0,0,0,...,5,5,3,6,4,2,0,0,Tuesday,10


In [ ]:
# seperate each row into one dish-day observation (restructured so we have a global model, not 1 model per dish)
DISHES = [
    "CALAMARI",
    "FISH",
    "PRAWNS",
    "CHICKEN",
    "KOFTA",
    "LAMB",
    "STEAK"
]

rows = []

for dish in DISHES:
    temp = pd.DataFrame()

    temp["DEMAND_DATE"] = df["DEMAND_DATE"]
    temp["DISH"] = dish

    # Target
    temp["DEMAND"] = df[dish]

    # Demand lags
    for lag in range(1, 8):
        temp[f"DEMAND_T{lag}"] = df[f"{dish}_DEMAND_T{lag}"]

    # Same-weekday historical averages
    for week in range(2, 5):
        temp[f"MEAN_SAME_WDAY_W{week}"] = \
            df[f"{dish}_MEAN_SAME_WDAY_DEMANDS_W{week}"]

    # Other dish-specific historical features
    for lag in range(2, 8):
        temp[f"CUM_DEMAND_T{lag}"] = \
            df[f"{dish}_CUM_DEMAND_T{lag}"]

    temp["HML_DEMAND_T7"] = \
        df[f"{dish}_HML_DEMAND_T7"]

    temp["NO_DAYS_ABOVE_7D_MEAN"] = \
        df[f"{dish}_NO_DAYS_ABOVE_7D_MEAN"]

    temp["NO_DAYS_BELOW_7D_MEAN"] = \
        df[f"{dish}_NO_DAYS_BELOW_7D_MEAN"]

    rows.append(temp)

long_df = pd.concat(rows, ignore_index=True)

long_df.shape

(5320, 22)

In [12]:
long_df.head()

,DEMAND_DATE,DISH,DEMAND,DEMAND_T1,DEMAND_T2,DEMAND_T3,DEMAND_T4,DEMAND_T5,DEMAND_T6,DEMAND_T7,...,MEAN_SAME_WDAY_W4,CUM_DEMAND_T2,CUM_DEMAND_T3,CUM_DEMAND_T4,CUM_DEMAND_T5,CUM_DEMAND_T6,CUM_DEMAND_T7,HML_DEMAND_T7,NO_DAYS_ABOVE_7D_MEAN,NO_DAYS_BELOW_7D_MEAN
0,2013-10-04,CALAMARI,6,6,5,4,7,4,9,3,...,3.0,11,15,22,26,35,38,6,3,4
1,2013-10-05,CALAMARI,8,6,6,5,4,7,4,9,...,9.0,12,17,21,28,32,41,5,4,3
2,2013-10-06,CALAMARI,6,8,6,6,5,4,7,4,...,4.0,14,20,25,29,36,40,4,4,3
3,2013-10-07,CALAMARI,4,6,8,6,6,5,4,7,...,7.0,14,20,26,31,35,42,4,2,2
4,2013-10-08,CALAMARI,7,4,6,8,6,6,5,4,...,4.0,10,18,24,30,35,39,4,4,3


In [ ]:
# shared features
shared_cols = [
    "MONDAY",
    "TUESDAY",
    "WEDNESDAY",
    "THURSDAY",
    "FRIDAY",
    "SATURDAY",
    "SUNDAY",

    "MONTH_JAN",
    "MONTH_FEB",
    "MONTH_MAR",
    "MONTH_APR",
    "MONTH_MAY",
    "MONTH_JUN",
    "MONTH_JUL",
    "MONTH_AUG",
    "MONTH_SEP",
    "MONTH_OCT",
    "MONTH_NOV",
    "MONTH_DEC",

    "ISHOLIDAY",
    "WEEKEND",

    "WIND",
    "CLOUD_COVER",
    "PRECIPITATION",
    "SUNSHINE",
    "AIR_TEMPERATURE"
]

shared_df = df[
    ["DEMAND_DATE"] + shared_cols
].copy()

long_df = long_df.merge(
    shared_df,
    on="DEMAND_DATE",
    how="left"
)
# copy shared features into the same day observation per dish

In [14]:
print(long_df.shape)
long_df.head(10)

(5320, 48)


,DEMAND_DATE,DISH,DEMAND,DEMAND_T1,DEMAND_T2,DEMAND_T3,DEMAND_T4,DEMAND_T5,DEMAND_T6,DEMAND_T7,...,MONTH_OCT,MONTH_NOV,MONTH_DEC,ISHOLIDAY,WEEKEND,WIND,CLOUD_COVER,PRECIPITATION,SUNSHINE,AIR_TEMPERATURE
0,2013-10-04,CALAMARI,6,6,5,4,7,4,9,3,...,1,0,0,0,0,1.916667,7.666667,0.1,150,15.858333
1,2013-10-05,CALAMARI,8,6,6,5,4,7,4,9,...,1,0,0,0,1,2.738462,6.923077,10.7,0,13.192308
2,2013-10-06,CALAMARI,6,8,6,6,5,4,7,4,...,1,0,0,0,1,1.364286,8.000000,0.4,0,10.571429
3,2013-10-07,CALAMARI,4,6,8,6,6,5,4,7,...,1,0,0,0,0,2.316667,6.416667,0.0,176,13.333333
4,2013-10-08,CALAMARI,7,4,6,8,6,6,5,4,...,1,0,0,0,0,1.658333,8.000000,0.0,0,13.541667
5,2013-10-09,CALAMARI,7,7,4,6,8,6,6,5,...,1,0,0,0,0,2.083333,7.666667,0.0,17,14.025000
6,2013-10-10,CALAMARI,3,7,7,4,6,8,6,6,...,1,0,0,0,0,2.983333,6.750000,4.3,0,7.058333
7,2013-10-11,CALAMARI,5,3,7,7,4,6,8,6,...,1,0,0,0,0,1.641667,5.583333,1.3,70,7.225000
8,2013-10-12,CALAMARI,5,5,3,7,7,4,6,8,...,1,0,0,0,1,1.461538,7.153846,0.3,85,4.992308
9,2013-10-13,CALAMARI,1,5,5,3,7,7,4,6,...,1,0,0,0,1,2.057143,4.285714,0.0,244,9.357143


In [ ]:
# yay didn't introduce any missing values from merging
long_df.isna().sum().sort_values(ascending=False).head(20)

DEMAND_DATE          0
DISH                 0
DEMAND               0
DEMAND_T1            0
DEMAND_T2            0
DEMAND_T3            0
DEMAND_T4            0
DEMAND_T5            0
DEMAND_T6            0
DEMAND_T7            0
MEAN_SAME_WDAY_W2    0
MEAN_SAME_WDAY_W3    0
MEAN_SAME_WDAY_W4    0
CUM_DEMAND_T2        0
CUM_DEMAND_T3        0
CUM_DEMAND_T4        0
CUM_DEMAND_T5        0
CUM_DEMAND_T6        0
CUM_DEMAND_T7        0
HML_DEMAND_T7        0
dtype: int64

In [ ]:
# test split!


# list of unique ascending dates bc long_df each date shows up 7 times
dates = np.sort(long_df["DEMAND_DATE"].unique())

# split into 70% for training, 15% for validating, 15% for testing
# ordered splits, trained on older data and tests using newer data
train_end = int(len(dates) * 0.70)
val_end = int(len(dates) * 0.85)

train_dates = dates[:train_end]
val_dates = dates[train_end:val_end]
test_dates = dates[val_end:]


# separate dataset into train, val, and test using the different dates
train_df = long_df[
    long_df["DEMAND_DATE"].isin(train_dates)
].copy()

val_df = long_df[
    long_df["DEMAND_DATE"].isin(val_dates)
].copy()

test_df = long_df[
    long_df["DEMAND_DATE"].isin(test_dates)
].copy()


print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

for name, split in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        split["DEMAND_DATE"].min(),
        "→",
        split["DEMAND_DATE"].max()
    )

Train: 3724
Validation: 798
Test: 798
Train 2013-10-04 00:00:00 → 2015-03-24 00:00:00
Validation 2015-03-25 00:00:00 → 2015-07-16 00:00:00
Test 2015-07-17 00:00:00 → 2015-11-07 00:00:00


In [19]:
TARGET = "DEMAND"

X_train = train_df.drop(
    columns=["DEMAND", "DEMAND_DATE"]
)

y_train = train_df["DEMAND"]

X_val = val_df.drop(
    columns=["DEMAND", "DEMAND_DATE"]
)

y_val = val_df["DEMAND"]

In [23]:
# encoding time!
categorical_features = ["DISH"]

numeric_features = [
    col for col in X_train.columns
    if col != "DISH"
]

preprocessor = ColumnTransformer([
    (
        "dish",     #dish becomes indicator variable
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    (
        "numeric",  # numeric columns get scaled (ridge penalizes coefficient sizes otherwise)
        StandardScaler(),
        numeric_features
    )
])

In [ ]:
# naive baseline for time series forecasting is next value = last observed value
# seasonal naive: use next value = same value last week

naive_val_pred = val_df["DEMAND_T1"]
seasonal_val_pred = val_df["DEMAND_T7"]

naive_mae = mean_absolute_error(
    y_val,
    naive_val_pred
)

naive_rmse = root_mean_squared_error(
    y_val,
    naive_val_pred
)

seasonal_mae = mean_absolute_error(
    y_val,
    seasonal_val_pred
)

seasonal_rmse = root_mean_squared_error(
    y_val,
    seasonal_val_pred
)

baseline_comparison = pd.DataFrame([
    {
        "Model": "Naive",
        "MAE": naive_mae,
        "RMSE": naive_rmse
    },
    {
        "Model": "Seasonal Naive",
        "MAE": seasonal_mae,
        "RMSE": seasonal_rmse
    }
])

baseline_comparison
# seasonal > naive as a baseline, but will compare both as a benchmark

,Model,MAE,RMSE
0,Naive,7.644110,11.497848
1,Seasonal Naive,6.394737,9.546155


In [25]:
# ridge model!
ridge_model = Pipeline([
    ("preprocessing", preprocessor),
    ("ridge", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](46,)","['DISH','DEMAND_T1','DEMAND_T2',...,'PRECIPITATION','SUNSHINE', 'AIR_TEMPERATURE']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,46
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('dish', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This 

In [29]:
ridge_val_pred = ridge_model.predict(X_val)

ridge_mae = mean_absolute_error(
    y_val,
    ridge_val_pred
)

ridge_rmse = root_mean_squared_error(
    y_val,
    ridge_val_pred
)

print("Ridge MAE:", ridge_mae)
print("Ridge RMSE:", ridge_rmse)

Ridge MAE: 4.954089116003066
Ridge RMSE: 7.092024393249851


In [30]:
# ridge v baselines
model_comparison = pd.DataFrame([
    {
        "Model": "Naive T-1",
        "MAE": naive_mae,
        "RMSE": naive_rmse
    },
    {
        "Model": "Seasonal Naive T-7",
        "MAE": seasonal_mae,
        "RMSE": seasonal_rmse
    },
    {
        "Model": "Ridge",
        "MAE": ridge_mae,
        "RMSE": ridge_rmse
    }
])

model_comparison

,Model,MAE,RMSE
0,Naive T-1,7.644110,11.497848
1,Seasonal Naive T-7,6.394737,9.546155
2,Ridge,4.954089,7.092024


In [ ]:
# ridge did good! now to check performance per dish cuz is it just doing well on some and worse on others?
val_results = val_df.copy()

val_results["RIDGE_PRED"] = ridge_val_pred

,Dish,Naive_MAE,Naive_RMSE,Seasonal_MAE,Seasonal_RMSE,Ridge_MAE,Ridge_RMSE
0,CALAMARI,2.526316,3.556561,2.245614,3.000000,2.415357,3.298067
1,FISH,2.728070,3.735792,2.438596,3.446330,2.637530,3.623542
2,PRAWNS,5.271930,6.843488,4.991228,5.899004,3.770406,4.719307
3,CHICKEN,13.184211,18.669564,10.517544,15.572412,7.110726,10.823772
4,KOFTA,8.263158,10.695137,6.991228,9.118614,5.551359,6.712810
5,LAMB,12.921053,16.233223,10.210526,12.372862,7.645877,9.567141
6,STEAK,8.614035,11.203070,7.368421,10.173064,5.547369,7.214740


In [ ]:
# wape (weighted abs. error percentage), lower is better
# 20% wape = total absolute forcasting error is around 20% of total actual demand
def wape(actual, predicted):
    return (
        np.abs(actual - predicted).sum()
        / np.abs(actual).sum()
    ) * 100

In [36]:
# mase (mean absolute scaled error) shows how large errors are relative to baseline (will use seasonal naive as baseline)
# lower is better, mase < 1 = better than baseline, mase = 0 means its a perfect prediction
def mase(actual, predicted, train_actual, train_seasonal_lag):
    scale = np.mean(
        np.abs(train_actual - train_seasonal_lag)
    )

    return mean_absolute_error(
        actual,
        predicted
    ) / scale

In [37]:
dish_comparison = []

for dish in DISHES:

    # Validation rows for this dish
    dish_df = val_results[
        val_results["DISH"] == dish
    ]

    # Training rows for this dish
    dish_train = train_df[
        train_df["DISH"] == dish
    ]

    actual = dish_df["DEMAND"]

    naive_pred = dish_df["DEMAND_T1"]
    seasonal_pred = dish_df["DEMAND_T7"]
    ridge_pred = dish_df["RIDGE_PRED"]

    train_actual = dish_train["DEMAND"]
    train_seasonal_lag = dish_train["DEMAND_T7"]

    dish_comparison.append({
        "Dish": dish,

        "Naive_MAE":
            mean_absolute_error(actual, naive_pred),

        "Naive_RMSE":
            root_mean_squared_error(actual, naive_pred),

        "Naive_WAPE":
            wape(actual, naive_pred),

        "Naive_MASE":
            mase(
                actual,
                naive_pred,
                train_actual,
                train_seasonal_lag
            ),

        "Seasonal_MAE":
            mean_absolute_error(actual, seasonal_pred),

        "Seasonal_RMSE":
            root_mean_squared_error(actual, seasonal_pred),

        "Seasonal_WAPE":
            wape(actual, seasonal_pred),

        "Seasonal_MASE":
            mase(
                actual,
                seasonal_pred,
                train_actual,
                train_seasonal_lag
            ),

        "Ridge_MAE":
            mean_absolute_error(actual, ridge_pred),

        "Ridge_RMSE":
            root_mean_squared_error(actual, ridge_pred),

        "Ridge_WAPE":
            wape(actual, ridge_pred),

        "Ridge_MASE":
            mase(
                actual,
                ridge_pred,
                train_actual,
                train_seasonal_lag
            )
    })

dish_comparison = pd.DataFrame(dish_comparison)

dish_comparison

,Dish,Naive_MAE,Naive_RMSE,Naive_WAPE,Naive_MASE,Seasonal_MAE,Seasonal_RMSE,Seasonal_WAPE,Seasonal_MASE,Ridge_MAE,Ridge_RMSE,Ridge_WAPE,Ridge_MASE
0,CALAMARI,2.526316,3.556561,64.864865,0.916780,2.245614,3.000000,57.657658,0.814916,2.415357,3.298067,62.015935,0.876514
1,FISH,2.728070,3.735792,63.211382,0.963701,2.438596,3.446330,56.504065,0.861443,2.637530,3.623542,61.113499,0.931717
2,PRAWNS,5.271930,6.843488,45.221971,1.211519,4.991228,5.899004,42.814146,1.147012,3.770406,4.719307,32.342081,0.866460
3,CHICKEN,13.184211,18.669564,41.738406,1.515885,10.517544,15.572412,33.296307,1.209279,7.110726,10.823772,22.511045,0.817572
4,KOFTA,8.263158,10.695137,45.507246,1.144196,6.991228,9.118614,38.502415,0.968072,5.551359,6.712810,30.572699,0.768694
5,LAMB,12.921053,16.233223,36.023478,1.350757,10.210526,12.372862,28.466618,1.067400,7.645877,9.567141,21.316459,0.799294
6,STEAK,8.614035,11.203070,40.595287,1.168154,7.368421,10.173064,34.725093,0.999235,5.547369,7.214740,26.143038,0.752282


In [ ]:
# metrics broken down (all want lower metrics)
# MAE: how many units on average is prediction off by
# RMSE: like MAE, but penalizes big mistakes more
# WAPE: how large is the error relative to demand? (error of 2 on a dish that has a demand of 4 is worse than a dish w/ a demand of 30)\
# MASE: performance against forecasting benchmark? 1 = performs the same, <1 means performs better

# Ridge outperformed naive baselines overall, with largest improvements on higher-demand dishes (chicken, lamb, steak). Seasonal naive remained stronger for low-volume dishes (calamari, fish).
# Ridge achieved MASE < 1 for all dishes, WAPE shows low volume dishes were harder to forecast accurately.